# Physical AI 基础篇 智能机器人课程 2026 Spring 第1讲练习资料
## 通过 LeRobot × SmolVLA × LIBERO 学习 VLA 推理流程

本 notebook 使用 Hugging Face 的轻量 VLA 模型 **SmolVLA** 的 LIBERO checkpoint `HuggingFaceVLA/smolvla_libero`，在机器人操作基准 **LIBERO** 上体验从
**推理 → 评估 → 可视化** 的完整流程。

这里要特别留意 VLA 的数据流：观测（图像、proprio 和语言指令）先经过预处理，模型一次预测一段 action chunk，再由后处理把动作还原为环境使用的尺度。这样可以把“看见什么”“理解什么”和“如何动作”串在同一条可检查的链路中。


## 检查 GPU

首先确认 Colab 分配了 **T4 GPU**。
如果 `nvidia-smi` 输出中出现 `Tesla T4` 和约 `15360MiB`（约 16GB）显存，就可以继续。


In [ ]:
!nvidia-smi

## 安装依赖

安装带有 **`smolvla`（SmolVLA）** 和 **`libero`（LIBERO 环境）** extra 的 LeRobot。

（之后需要重启 session。）


In [ ]:
!apt-get update
!apt-get install -y \
    libegl1 libegl1-mesa-dev libgles2-mesa-dev \
    libglfw3 libosmesa6-dev libgl1-mesa-glx

!pip install "lerobot[smolvla,libero]==0.5.1"

## 什么是 SmolVLA

**SmolVLA** 是 Hugging Face 在 2025 年发布的轻量 **Vision-Language-Action（视觉-语言-动作，VLA）** 模型。它的目标是做到**可以在单张 GPU 上训练，也能在消费级 GPU 或 CPU 上运行**。

### 训练数据

- 使用 **LeRobot 社区公开的 487 个真实机器人数据集**进行预训练
- 主要依靠**社区公开数据**，而不是大企业的私有超大规模数据；与同类模型相比，训练数据规模小了一个数量级以上

### 性能（LIBERO 基准，论文报告值）

| 模型 | 参数量 | 平均成功率 |
| --- | --- | --- |
| Octo | 0.09B | 75.1% |
| OpenVLA | 7B | 76.5% |
| π0 | 3.3B | 86.0% |
| **SmolVLA** | **0.45B** | **87.3%** |

本 notebook 使用的 `HuggingFaceVLA/smolvla_libero` 是针对 **LIBERO 训练的 SmolVLA checkpoint**。

参考：
- 论文 https://arxiv.org/abs/2506.01844
- 解读博客 https://huggingface.co/blog/smolvla


## 什么是 LIBERO

**LIBERO** 是常用于评估机器人策略性能的**操作任务基准**（Liu et al., NeurIPS 2023）。它在仿真器中定义抓取、放置、打开等操作任务。

### 组成

- 基于物理仿真器 **MuJoCo** 的 **Franka Emika Panda** 机械臂
- **5 个任务 suite（共 130 个任务）**，每个任务都带有人类示范（遥操作）

| Suite | 任务数 | 评估变化 |
| --- | --- | --- |
| `libero_spatial` | 10 | 物体的放置位置（空间关系） |
| `libero_object` | 10 | 目标物体不同 |
| `libero_goal` | 10 | 目标不同 |
| `libero_10`（LIBERO-Long） | 10 | 长时间、复合任务 |
| `libero_90` | 90 | 多样的短任务 |

### 输入与输出

- **观测（observation）**
  - `observation.state`：8 维 proprio（末端位置与姿态 6 + 夹爪两指 2）
  - `observation.images.image`：主相机（俯视）
  - `observation.images.image2`：腕部相机
  - `task`：自然语言指令，例如 "pick up the black bowl and put it on the plate"
- **动作（action）**：7 维（末端位姿增量 6 + 夹爪开合 1）

### 评估指标

- 对每个任务运行多个 episode 的 **rollout**，测量**任务成功率（success rate）**。
- “成功”由每个任务的完成条件（目标物体是否到达目标位置或状态）自动判断。

官方文档：
- https://huggingface.co/docs/lerobot/libero
- https://libero-project.github.io/main


## 什么是 LeRobot

**LeRobot** 是 Hugging Face 开发的、面向**真实机器人学习的开源框架**。模型等组件全部使用 PyTorch 编写。
它能够一站式处理**数据、模型、训练、评估和真实机器人控制**，目前许多项目都使用 LeRobot 格式的数据集。

### 主要组件

- **`LeRobotDataset`**：以统一格式处理机器人 episode（图像、proprio、动作、语言指令）。它可以从 HF Hub 共享和下载，本 notebook 使用的 `HuggingFaceVLA/libero` 也属于这种格式。
- **机器人策略**：包含 ACT、Diffusion Policy、π0、**SmolVLA** 等多种机器人策略实现。
- **环境**：LIBERO、PushT 等仿真环境的封装。

官方仓库：https://github.com/huggingface/lerobot ／ 文档：https://huggingface.co/docs/lerobot


## 加载模型

使用 `SmolVLAPolicy.from_pretrained` 加载 checkpoint **`HuggingFaceVLA/smolvla_libero`**（接口风格与 `transformers` 等库相似）。

同时用 `make_pre_post_processors` 构建**预处理（preprocessor）/后处理（postprocessor）**流水线。这些步骤是推理所必需的：

- **preprocessor**：观测字典键转换 → 增加 batch 维度 → 对语言指令分词 → 转移到设备 → **归一化**
- **postprocessor**：对模型输出动作做**反归一化** → 转移到 CPU

归一化统计量保存在 checkpoint 中，因此把 checkpoint ID 作为 `make_pre_post_processors` 的**第二个参数**，就能加载与训练时相同的统计量。这正是“训练时和推理时保持处理一致”的关键。


In [ ]:
import os
import torch
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
from lerobot.policies.factory import make_pre_post_processors

os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"

MODEL_ID = "HuggingFaceVLA/smolvla_libero"
device = "cuda"

# checkpoint-（首次是几百 MB 的下载走）
model = SmolVLAPolicy.from_pretrained(MODEL_ID)
model = model.to(device)
model.eval()

# 预处理・后处理流水线（归一化統計是checkpoint从読込）
preprocess, postprocess = make_pre_post_processors(
    model.config,
    MODEL_ID,
    preprocessor_overrides={"device_processor": {"device": device}},
)

torch.cuda.synchronize()
!nvidia-smi

## 推理演示

从 LIBERO 数据集 **`HuggingFaceVLA/libero`** 中取出**一个 episode**，把第一帧输入模型并输出一个 **action chunk**。


In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset

# 1 episode仅-
dataset = LeRobotDataset("HuggingFaceVLA/libero", episodes=[0])

# episode 0 的先頭帧 index 获取
ep = 0
from_idx = dataset.meta.episodes["dataset_from_index"][ep]
frame = dataset[from_idx]

observation = {
    "observation.state": frame["observation.state"],
    "observation.images.image": frame["observation.images.image"],
    "observation.images.image2": frame["observation.images.image2"],
    "task": frame["task"],
}

print("=== 输入（观测）的 shape ===")
print("state       :", tuple(observation["observation.state"].shape))
print("image       :", tuple(observation["observation.images.image"].shape), "(C, H, W)")
print("image2      :", tuple(observation["observation.images.image2"].shape), "(C, H, W)")
print("task (指令):", repr(observation["task"]))

显示输入图像（主相机和腕部相机），确认模型正在“看见”什么。


In [ ]:
import matplotlib.pyplot as plt

# 几据集的图像是 (C, H, W) 因此显示用到 (H, W, C) 方向 permute
img_main = observation["observation.images.image"].permute(1, 2, 0).cpu().numpy()
img_wrist = observation["observation.images.image2"].permute(1, 2, 0).cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img_main); axes[0].set_title("agentview (main)"); axes[0].axis("off")
axes[1].imshow(img_wrist); axes[1].set_title("eye-in-hand (wrist)"); axes[1].axis("off")
plt.suptitle(observation["task"])
plt.tight_layout()
plt.show()

执行预处理 → 推理 → 后处理流程。`predict_action_chunk` 获取**完整 action chunk**，`select_action` 获取**单个时间步的动作**，并分别检查它们的 shape。


In [ ]:
model.reset()  # 清空每个 episode 开始时的 action queue
processed = preprocess(observation)                  # 预处理（batch维度付与・分词・归一化等）
action_chunk = model.predict_action_chunk(processed) # (B, n_action_steps, action_dim)
print("=== 输出（action chunk） ===")
print("action_chunk:", tuple(action_chunk.shape), "= (batch, 步几, 动作维度)")
print("action值的示示示示例:", [round(x, 3) for x in action_chunk[0][0].tolist()])
print("postprocess后:", [round(x, 3) for x in postprocess(action_chunk)[0][0].tolist()])


- `action_chunk` 的 `(B, n_action_steps, 7)` 表示：一次推理会预测未来 `n_action_steps` 个时间步的动作。这称为 VLA 的 **action chunking**。
- 在实际控制中，这个 chunk 会先放入内部队列，由 `select_action` 每次取出一个时间步；队列耗尽后再进行推理。


## 基准评估

LeRobot 通常可以通过 `lerobot-eval` 命令在 LIBERO 仿真器上实际 rollout，并测量**任务成功率**。
但本次不直接依赖黑盒命令，而是手写评估脚本，以便看清评估内部做了什么。


In [ ]:
EVAL_OUT = "./eval_out"

# 以下注意释掉执行与， lerobot-eval 执行与也可能
# cmd = (
#     "lerobot-eval"
#     f" --policy.path={MODEL_ID}"
#     " --policy.device=cuda"
#     " --policy.n_action_steps=10"
#     " --env.type=libero"
#     " --env.task=libero_spatial"
#     ' --env.task_ids="[0]"'
#     " --eval.n_episodes=3"
#     " --eval.batch_size=1"
#     " --env.max_parallel_tasks=1"
#     f" --output_dir={EVAL_OUT}"
# )
# print(cmd, "\n")
# !{cmd}

In [ ]:
import time
import numpy as np
import torch
from libero.libero import benchmark
from lerobot.envs.libero import LiberoEnv
import imageio

# ===== 评估配置 =====
SUITE_NAME = "libero_spatial"   # 评估执行suite
TASK_ID = 0                     # suite内的任务番号
N_EPISODES = 3                  # 评估episode几

# action chunk
model.config.n_action_steps = 10

# ===== 环境创建 =====
suite = benchmark.get_benchmark_dict()[SUITE_NAME]()
env = LiberoEnv(
    task_suite=suite,
    task_id=TASK_ID,
    task_suite_name=SUITE_NAME,
    obs_type="pixels_agent_pos",
    control_mode="relative",
    observation_width=256,
    observation_height=256,
    init_states=True,
    n_envs=1,
)
max_steps = env._max_episode_steps
print(f"任务指令: {env.task_description}")
print(f"1 episode的最大step: {max_steps}\n")


def quat2axisangle(quat):  # quat: torch (4,) 順序是 (x, y, z, w)
    w = quat[3].clamp(-1.0, 1.0)
    den = torch.sqrt(torch.clamp(1.0 - w * w, min=0.0))
    if den < 1e-10:
        return torch.zeros(3, dtype=torch.float32)
    return (quat[:3] / den * (2.0 * torch.acos(w))).float()

def to_chw(img_hwc):
    img = np.ascontiguousarray(img_hwc[::-1, ::-1])
    return torch.from_numpy(img).permute(2, 0, 1).float() / 255.0

def build_observation(obs):
    rs = obs["robot_state"]
    eef_pos = torch.as_tensor(rs["eef"]["pos"], dtype=torch.float32)       # (3,)
    eef_quat = torch.as_tensor(rs["eef"]["quat"], dtype=torch.float32)     # (4,) (x,y,z,w)
    gripper = torch.as_tensor(rs["gripper"]["qpos"], dtype=torch.float32)  # (2,)
    state = torch.cat([eef_pos, quat2axisangle(eef_quat), gripper])        # (8,)
    return {
        "observation.state": state,
        "observation.images.image": to_chw(obs["pixels"]["image"]),
        "observation.images.image2": to_chw(obs["pixels"]["image2"]),
        "task": env.task_description,
    }


results = []         # 每个 episode 的成功 / 失败
rollout_frames = []  # 视频帧
t0 = time.time()

for ep in range(N_EPISODES):
    env.init_state_id = ep   # 初期状態固定（lerobot 与相同 0,1,2,... 使）
    obs, info = env.reset()
    model.reset()            # 清空 action chunk queue
    success = False

    for t in range(max_steps):
        print(f"Episode {ep}: step {t + 1}")

        rollout_frames.append(np.ascontiguousarray(obs["pixels"]["image"][::-1, ::-1]))

        # 预处理 → 1 step分的action获取 → 后处理（逆归一化）
        processed = preprocess(build_observation(obs))
        with torch.inference_mode():
            action = model.select_action(processed)
        action = postprocess(action).squeeze(0).cpu().numpy()

        obs, reward, terminated, truncated, info = env.step(action)
        success = success or bool(info.get("is_success", False))
        if terminated:
            break

    results.append(success)
    print(f"  Episode {ep}: {'✅ 成功' if success else '❌ 失败'}  ({t + 1} steps)")

    # Save the video
    VIDEO_PATH = f"rollout_ep{ep}.mp4"
    imageio.mimsave(VIDEO_PATH, rollout_frames, fps=30)
    rollout_frames = []

env.close()

success_rate = 100.0 * sum(results) / len(results)
print("\n=== 评估结果 ===")
print(f"成功率: {success_rate:.1f}%  ({sum(results)}/{len(results)} episodes)")
print(f"耗时: {time.time() - t0:.1f} 秒")

In [ ]:
from IPython.display import Video, display

for ep in range(N_EPISODES):
    VIDEO_PATH = f"rollout_ep{ep}.mp4"
    print(f"episode {ep}: {VIDEO_PATH}")
    display(Video(VIDEO_PATH, embed=True, width=480))